In [2]:
import warnings
from statsmodels.tsa.stattools import adfuller
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss
import sklearn.model_selection as ms
import yfinance as yf

# Tắt cảnh báo có chọn lọc
warnings.filterwarnings("ignore", category=UserWarning)

print("==================================================")
print(" KHỞI CHẠY HỆ THỐNG ĐỊNH LƯỢNG KRW/VND (V5.0 - PRO)")
print("==================================================")

# =====================================================================
# TẦNG 1: DATA & CLEANING LAYER (Lọc Stale/Spike trên từng chân độc lập)
# =====================================================================
print("\n--- [Tầng 1] Tải dữ liệu 5 năm và làm sạch chân độc lập ---")

usd_krw = yf.download("KRW=X", period="5y", interval="1d", auto_adjust=True)[
    "Close"
]
usd_vnd = yf.download("VND=X", period="5y", interval="1d", auto_adjust=True)[
    "Close"
]

if isinstance(usd_krw, pd.DataFrame):
  usd_krw = usd_krw.iloc[:, 0]
if isinstance(usd_vnd, pd.DataFrame):
  usd_vnd = usd_vnd.iloc[:, 0]

# Loại bỏ bar chưa đóng của ngày hiện tại (nếu có)
today_date = pd.Timestamp.today().normalize()
if usd_krw.index[-1] >= today_date:
  usd_krw = usd_krw.iloc[:-1]
if usd_vnd.index[-1] >= today_date:
  usd_vnd = usd_vnd.iloc[:-1]

df = pd.DataFrame({"USD_KRW": usd_krw, "USD_VND": usd_vnd}).dropna()

# Lọc stale data và spike trên từng chân độc lập
for col in ["USD_KRW", "USD_VND"]:
  df = df[df[col].diff() != 0]
  df = df[df[col].pct_change().abs() < 0.02]

raw_prices = df["USD_VND"] / df["USD_KRW"]

if raw_prices.index.tz is not None:
  raw_prices.index = raw_prices.index.tz_localize(None)
raw_prices.index = pd.to_datetime(raw_prices.index).normalize()

log_prices = np.log(raw_prices)
ROUND_TRIP_COST = 0.0040  # 0.4% chi phí khứ hồi cơ sở


# =====================================================================
# TẦNG 2: OPTIMIZED FFD LAYER (Không cắt cụt lag, thres chuẩn 1e-4)
# =====================================================================
print("\n--- [Tầng 2] Quét tìm bậc d tối ưu trên Log-Prices ---")


def get_weights_ffd(d, thres=1e-4):
  w = [1.0]
  k = 1
  while True:
    w_ = -w[-1] / k * (d - k + 1)
    if abs(w_) < thres:
      break
    w.append(w_)
    k += 1
  return np.array(w[::-1])


def fractional_diff_ffd(series, d, thres=1e-4):
  w = get_weights_ffd(d, thres)
  width = len(w) - 1
  df_res = pd.Series(index=series.index, dtype=float)
  for iloc in range(width, len(series)):
    loc = series.index[iloc]
    sub_series = series.iloc[iloc - width : iloc + 1]
    df_res.loc[loc] = np.dot(w, sub_series.values)
  return df_res.dropna()


best_d = 0.05
tested_ds = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0]
for d_test in tested_ds:
  test_series = fractional_diff_ffd(log_prices, d=d_test)
  if len(test_series) > 100:
    adf_result = adfuller(test_series)
    if adf_result[1] < 0.05:
      best_d = d_test
      break
    best_d = d_test

print(f" -> Bậc d tối ưu được chọn: {best_d}")
stationary_series = fractional_diff_ffd(log_prices, d=best_d)


# =====================================================================
# TẦNG 3: TRUE META-MODEL LAYER (Thực thi tại t+1 & Chống Overfit)
# =====================================================================
print(
    "\n--- [Tầng 3] Xây dựng đặc trưng và gán nhãn với điểm khớp lệnh t+1 ---"
)

clean_features = pd.DataFrame(index=log_prices.index, dtype=float)
clean_features["Log_Price"] = log_prices
clean_features["Frac_Diff"] = stationary_series
clean_features["Rolling_Mean"] = stationary_series.rolling(window=5).mean()
clean_features["Rolling_Std"] = stationary_series.rolling(window=5).std()
clean_features["Return_1d"] = log_prices.diff()

primary_signal = np.where(log_prices.diff(10) > 0, 1, -1)
clean_features["Primary_Side"] = pd.Series(
    np.where(log_prices.diff(10).isna(), np.nan, primary_signal),
    index=log_prices.index,
)


def get_daily_vol(close, span=30):
  return close.pct_change().ewm(span=span).std().dropna()


daily_vol = get_daily_vol(raw_prices)


def apply_triple_barrier_t1(prices_series, events, pt_sl):
  out = pd.DataFrame(index=events.index)
  out["ret"] = np.nan
  out["bin"] = 0
  for loc, t1 in events["t1"].items():
    # Thực thi tại loc+1 (t+1) sau khi tín hiệu tính ở loc (t)
    loc_idx = prices_series.index.get_loc(loc)
    if loc_idx + 1 >= len(prices_series):
      continue
    exec_loc = prices_series.index[loc_idx + 1]

    path = prices_series.loc[exec_loc:t1]
    if len(path) <= 1:
      continue
    base_val = prices_series.loc[exec_loc]
    ret = (path / base_val) - 1
    side = events.loc[loc, "side"]
    ret_directional = ret * side
    target = max(events.loc[loc, "trgt"], 0.001)

    pt = target * pt_sl[0]
    sl = -target * pt_sl[1]

    hit_pt = ret_directional[ret_directional >= pt].index.min()
    hit_sl = ret_directional[ret_directional <= sl].index.min()

    if pd.notna(hit_pt) and (pd.isna(hit_sl) or hit_pt < hit_sl):
      out.loc[loc, "bin"] = 1
      out.loc[loc, "ret"] = ret.loc[hit_pt] * side
    elif pd.notna(hit_sl) and (pd.isna(hit_pt) or hit_sl < hit_pt):
      out.loc[loc, "bin"] = -1
      out.loc[loc, "ret"] = ret.loc[hit_sl] * side
    else:
      out.loc[loc, "bin"] = 0
      out.loc[loc, "ret"] = ret.iloc[-1] * side
  return out.dropna(subset=["ret"])


holding_period = 15
dropped_features = clean_features.dropna()
events_index = dropped_features.index[:-holding_period]
events = pd.DataFrame(index=events_index)
events["trgt"] = daily_vol.loc[events.index]
events["side"] = dropped_features.loc[events.index, "Primary_Side"]
events["t1"] = raw_prices.index[
    [
        min(
            raw_prices.index.get_loc(idx) + holding_period,
            len(raw_prices) - 1,
        )
        for idx in events.index
    ]
]

labels = apply_triple_barrier_t1(raw_prices, events, pt_sl=[2.0, 2.0])

valid_idx = labels.index.intersection(dropped_features.index)
X = dropped_features.loc[
    valid_idx,
    ["Frac_Diff", "Rolling_Mean", "Rolling_Std", "Return_1d", "Primary_Side"],
]
meta_y = (labels.loc[valid_idx, "ret"] > 0).astype(int)

print(f" -> Số lượng mẫu dữ liệu sạch sau xử lý: {len(X)}")
print(f" -> Phân phối Meta-y: {meta_y.value_counts().to_dict()}")


# =====================================================================
# TẦNG 4: RIGOROUS VALIDATION LAYER (Embargo >= Holding Period & Calibrated CV)
# =====================================================================
print(
    "\n--- [Tầng 4] Kiểm định chéo Purged CV (Embargo 3% ~ 15+ hàng) & OOF"
    " Evaluation ---"
)


class PurgedKFold(ms.KFold):

  def __init__(self, n_splits=5, t1=None, embargo_pct=0.03):
    super(PurgedKFold, self).__init__(
        n_splits=n_splits, shuffle=False, random_state=None
    )
    self.t1 = t1
    self.embargo_pct = embargo_pct

  def split(self, X, y=None, groups=None):
    if self.t1 is None:
      raise ValueError("Cần chuỗi t1")
    for train_index, test_index in super().split(X, y):
      test_start = X.index[test_index[0]]
      test_end = X.index[test_index[-1]]
      train_times = self.t1.loc[X.index[train_index]]

      banned_indices = train_times[
          (train_times.index <= test_end) & (train_times >= test_start)
      ].index
      train_index = np.array(
          [i for i in train_index if X.index[i] not in banned_indices]
      )

      embargo_idx = int(X.shape[0] * self.embargo_pct)
      if embargo_idx > 0:
        test_end_loc = X.index.get_loc(test_end)
        embargo_end_loc = min(test_end_loc + embargo_idx, X.shape[0])
        embargo_dates = X.index[test_end_loc + 1 : embargo_end_loc]
        train_index = np.array(
            [i for i in train_index if X.index[i] not in embargo_dates]
        )
      yield train_index, test_index


t1_series = events["t1"].loc[X.index]
cv = PurgedKFold(n_splits=5, t1=t1_series, embargo_pct=0.03)

# Thu thập OOF (Out-of-Fold) probabilities từ pipeline đã hiệu chuẩn nội bộ
oof_probs = np.zeros(len(X))
splits_list = list(cv.split(X))

brier_scores = []
baseline_briers = []

for train_idx, test_idx in splits_list:
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = meta_y.iloc[train_idx], meta_y.iloc[test_idx]

  base_fold = RandomForestClassifier(
      n_estimators=100,
      max_depth=3,
      min_samples_leaf=30,
      random_state=42,
      class_weight="balanced",
  )
  # Hiệu chuẩn ngay bên trong fold để đánh giá chính xác pipeline thực tế
  cal_fold = CalibratedClassifierCV(
      base_fold, method="sigmoid", cv=list(ms.KFold(n_splits=3).split(X_train))
  )
  cal_fold.fit(X_train, y_train)

  probs = cal_fold.predict_proba(X_test)[:, 1]
  oof_probs[test_idx] = probs

  brier_scores.append(brier_score_loss(y_test, probs))
  p_mean = y_train.mean()
  baseline_briers.append(
      brier_score_loss(y_test, np.full_like(y_test, p_mean, dtype=float))
  )

valid_oof_mask = oof_probs > 0  # Các mẫu nằm ngoài train/test đầu tiên nếu có
print(
    f" -> Calibrated Pipeline Brier Score (OOF):"
    f" {np.mean(brier_scores):.4f} (± {np.std(brier_scores):.4f})"
)
print(
    f" -> Brier Baseline (p): {np.mean(baseline_briers):.4f} (±"
    f" {np.std(baseline_briers):.4f})"
)

# Fit mô hình sản phẩm toàn cục (Global Production Model)
global_base = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    min_samples_leaf=30,
    random_state=42,
    class_weight="balanced",
)
meta_model = CalibratedClassifierCV(
    global_base, method="sigmoid", cv=splits_list
)
meta_model.fit(X, meta_y)


# =====================================================================
# TẦNG 5: OOF BACKTEST, COST SENSITIVITY & LIVE INFERENCE
# =====================================================================
print("\n--- [Tầng 5] Backtest Out-of-Fold & Bảng nhạy cảm chi phí ---")

historical_gains = labels.loc[labels["ret"] > 0, "ret"].abs().mean()
historical_losses = labels.loc[labels["ret"] < 0, "ret"].abs().mean()
if pd.isna(historical_gains) or historical_gains == 0:
  historical_gains = 0.01
if pd.isna(historical_losses) or historical_losses == 0:
  historical_losses = 0.01

# In bảng nhạy cảm chi phí (Cost Sensitivity Table)
print("\n[BẢNG NHẠY CẢM CHI PHÍ & NGƯỠNG HÀO VỐN p*]")
print(f"Gain trung bình: {historical_gains*100:.2f}% | Loss trung bình: {historical_losses*100:.2f}%")
print(f"{'Cost (%)':<10} | {'p* Hòa vốn (%)':<15} | {'Số lệnh OOF (p > p*)':<20}")
print("-" * 51)
for c in [0.001, 0.002, 0.004, 0.008]:
  p_star = (historical_losses + c) / (historical_gains + historical_losses)
  n_trades = np.sum((oof_probs > p_star) & valid_oof_mask)
  print(f"{c*100:,.1f}%      | {p_star*100:,.1f}%          | {n_trades}")

# Đánh giá OOF Performance với Cost = 0.4%
p_breakeven = (historical_losses + ROUND_TRIP_COST) / (
    historical_gains + historical_losses
)
selected_trades = (oof_probs > (p_breakeven + 0.03)) & valid_oof_mask  # Thêm margin 3%
if np.sum(selected_trades) > 0:
  trade_rets = labels.loc[X.index[selected_trades], "ret"] - ROUND_TRIP_COST
  hit_rate = (trade_rets > 0).mean()
  mean_ret = trade_rets.mean()
  print(
      f"\n -> Kết quả Backtest OOF (với margin p* + 3%):"
      f" {np.sum(selected_trades)} lệnh | Hit Rate: {hit_rate*100:.1f}% | Lợi"
      f" nhuận thuần TB/lệnh: {mean_ret*100:.4f}%"
  )
else:
  print(
      "\n -> Kết quả Backtest OOF: Không có lệnh nào vượt qua ngưỡng p* + margin"
      " (Hệ thống đứng ngoài hoàn toàn)."
  )

# Tín hiệu Live cho phiên mới nhất
clean_features_fixed = clean_features.dropna()
current_data = clean_features_fixed.iloc[[-1]]
latest_date = current_data.index[0].strftime("%Y-%m-%d")
current_price = raw_prices.loc[current_data.index[0]]

pred_prob = meta_model.predict_proba(
    current_data[
        [
            "Frac_Diff",
            "Rolling_Mean",
            "Rolling_Std",
            "Return_1d",
            "Primary_Side",
        ]
    ]
)[:, 1][0]
current_side = current_data["Primary_Side"].values[0]

p_win = pred_prob
p_loss = 1 - pred_prob
expected_value = (
    (p_win * historical_gains)
    - (p_loss * historical_losses)
    - ROUND_TRIP_COST
)

print("\n" + "=" * 65)
print(f" BẢN TIN TRỰC TUYẾN INSTITUTIONAL - NGÀY: {latest_date}")
print("=" * 65)
print(f" Tỷ giá KRW/VND tham chiếu: {current_price:.4f}")
print(f" Hướng đi Primary Model: {'MUA (LONG)' if current_side == 1 else 'BÁN (SHORT)'}")
print(f" Xác suất thành công (p): {pred_prob * 100:.2f}%")
print(f" Ngưỡng hòa vốn p*: {p_breakeven * 100:.2f}%")
print(f" Kỳ vọng lợi nhuận thuần (EV): {expected_value * 100:.4f}%")

if expected_value > 0 and pred_prob > (p_breakeven + 0.03):
  print(" 🟢 TÍN HIỆU: ĐÈN XANH (Vượt p* + margin, đủ điều kiện giải ngân t+1)")
else:
  print(" 🔴 TÍN HIỆU: ĐÈN ĐỎ (Chưa đạt biên độ an toàn / Đứng ngoài)")
print("=" * 65)

 KHỞI CHẠY HỆ THỐNG ĐỊNH LƯỢNG KRW/VND (V5.0 - PRO)

--- [Tầng 1] Tải dữ liệu 5 năm và làm sạch chân độc lập ---


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed



--- [Tầng 2] Quét tìm bậc d tối ưu trên Log-Prices ---
 -> Bậc d tối ưu được chọn: 0.05

--- [Tầng 3] Xây dựng đặc trưng và gán nhãn với điểm khớp lệnh t+1 ---
 -> Số lượng mẫu dữ liệu sạch sau xử lý: 770
 -> Phân phối Meta-y: {0: 390, 1: 380}

--- [Tầng 4] Kiểm định chéo Purged CV (Embargo 3% ~ 15+ hàng) & OOF Evaluation ---
 -> Calibrated Pipeline Brier Score (OOF): 0.2546 (± 0.0038)
 -> Brier Baseline (p): 0.2540 (± 0.0021)

--- [Tầng 5] Backtest Out-of-Fold & Bảng nhạy cảm chi phí ---

[BẢNG NHẠY CẢM CHI PHÍ & NGƯỠNG HÀO VỐN p*]
Gain trung bình: 1.47% | Loss trung bình: 1.40%
Cost (%)   | p* Hòa vốn (%)  | Số lệnh OOF (p > p*)
---------------------------------------------------
0.1%      | 52.3%          | 143
0.2%      | 55.7%          | 6
0.4%      | 62.7%          | 0
0.8%      | 76.6%          | 0

 -> Kết quả Backtest OOF: Không có lệnh nào vượt qua ngưỡng p* + margin (Hệ thống đứng ngoài hoàn toàn).

 BẢN TIN TRỰC TUYẾN INSTITUTIONAL - NGÀY: 2026-11-03
 Tỷ giá KRW/VND tham c